# Nondimensionalization, one step at a time, with a proof at the end

Four models from a first course in mathematical biology. For each one we ask the same questions:

1. Which changes of units leave the equations unchanged? (the *scaling symmetries*)
2. Which dimensionless variables do they give, and what is the reduced model?
3. Could we have asked for something else, such as keeping $R_0$?
4. What has a proof assistant actually checked?

Run the cells in order. Nothing here needs Lean; the last section shows where the Lean proofs live.

In [ ]:
import pathlib, sys
try:
    import nondim
except ImportError:                      # running from a source checkout
    sys.path.insert(0, str(pathlib.Path.cwd().parent / "python"))
    sys.path.insert(0, str(pathlib.Path.cwd() / "python"))
    import nondim

from nondim import models
from nondim.scaling import analyze, nondimensionalize
from nondim.constraints import solve
from nondim.identifiability import analyze_identifiability
from nondim.cli import own_reduction
from nondim.catalog import CATALOG
from nondim.lean_export2 import certificate

## 1. Logistic growth

$$\frac{dN}{dt} = rN\left(1 - \frac{N}{K}\right)$$

Two things can be measured in different units: time and population size. Changing the unit of time
by a factor $\lambda$ multiplies $t$ by $\lambda$ and divides $r$ by $\lambda$. Changing the unit of
population multiplies $N$ and $K$ together. Those are the two scaling symmetries, and the tool finds them
from the exponents in the equation.

In [ ]:
logistic = models.logistic()
print(logistic)
print()
for line in analyze(logistic).describe_scalings():
    print(line)

Each scaling can be used to set one parameter to 1. Here that removes both parameters: every logistic
curve is the same curve after rescaling.

In [ ]:
print(nondimensionalize(logistic).summary())

## 2. SIR and the basic reproduction number

$$\frac{dS}{dt} = -\beta S I, \qquad \frac{dI}{dt} = \beta S I - \gamma I, \qquad \frac{dR}{dt} = \gamma I,
\qquad S + I + R = N$$

There are three parameters, $\beta$, $\gamma$, $N$, and again two scalings. So one parameter must
survive. Which one is a choice. Measuring time in units of the infectious period $1/\gamma$ and people as
a fraction of $N$ leaves $R_0 = \beta N / \gamma$.

The conservation law matters: it is the reason $N$ has to be rescaled together with $S$, $I$, $R$.
The model declares that as a *unit link*.

In [ ]:
sir = models.ALL["sir"]()
print(nondimensionalize(sir, normalize=["gamma", "N"]).summary())

### Asking for something specific

A request is answered *possible* (with the reduction) or *impossible* (with the reason).

In [ ]:
print(solve(sir, eliminate_at_least=2, keep=["beta"]))
print(solve(sir, eliminate_at_least=3))
print(solve(sir, at_most_params=1, want={"R0": "beta*N/gamma"}))
print(solve(sir, want={"force": "beta*N"}))

## 3. What the data cannot tell you

Suppose only reported cases $y = \rho I$ are observed, with an unknown reporting fraction $\rho$.
Doubling $S$ and $I$ while halving $\beta$ and $\rho$ changes nothing an observer can see. So $\beta$
and $\rho$ cannot be estimated separately from this data, however good it is.

This test only goes one way. A parameter it does not flag may still be impossible to estimate for another
reason; use a full identifiability tool for a complete answer.

In [ ]:
print(analyze_identifiability(models.sir_reported()).summary())

## 4. Predator and prey with a saturating appetite (Rosenzweig–MacArthur)

$$\frac{dN}{dt} = rN\left(1 - \frac{N}{K}\right) - \frac{aNP}{1 + ahN}, \qquad
\frac{dP}{dt} = \frac{eaNP}{1 + ahN} - mP$$

The right-hand sides are quotients, not polynomials. A scaling is a symmetry only if it gives every term
of the denominator $1 + ahN$ the same weight, which forces $ahN$ to be dimensionless.
Six parameters, three scalings, three parameters left.

In [ ]:
rm = own_reduction("rosenzweig_macarthur")
print(rm.summary())

## 5. What Lean checks

For each model the tool writes a file of theorems (in `lean/NondimVerify/Certified/`) that the Lean proof
assistant must prove. If the Python code above were wrong, that file would not compile. For SIR the main
statement is:

In [ ]:
text = certificate(own_reduction("sir"), "Own.SIR", CATALOG["sir"]["constraints"])
start = text.index("/-- **Main statement.**")
print(text[start:text.index("/-- **The same solution", start)].rstrip())

In words: *take any solution of the SIR equations with positive values; then $S/N$, $I/N$ and $R/N$,
followed in the time $\tau = \gamma t$, satisfy the reduced equations with the single parameter $R_0$.*

The same file also proves the answers to the requests of section 2, for example:

In [ ]:
for line in text.splitlines():
    if line.startswith(("-- Goal", "-- Answer", "theorem goal", "theorem cannot_remove_more")):
        print(line[:150])

## What this does not tell you

* Whether the equations you typed are the model you meant.
* Which nondimensionalization is the *useful* one. That depends on which quantities are small in your
  system, and it is a modelling judgement.
* Anything about solutions that become zero or negative, or models that are not built from polynomials
  and quotients of polynomials.

More: `docs/GUIDE.md` (how to use the tool), `docs/THEORY.md` (the theorems), `docs/BACKGROUND.md`
(the same calculations by hand).